# HDF5 format

> One `.h5` file in stable-worldmodel's layout: a dataset per column plus `ep_len` / `ep_offset`.

In [ ]:
#| default_exp data.hdf5

In [ ]:
#| hide
from nbdev.showdoc import *

The file layout is the one of stable-worldmodel's `hdf5` format, so stable-worldmodel's own
`HDF5Dataset` can read datasets collected here (and the other way around). Additionally:

* the dataset metadata (env name and kwargs, number of agents) is stored as file attributes,
  ignored by readers that do not know about it
* columns are compressed (``gzip`` by default): about 4 KiB per step for two agents' rendered ``pixels``, instead of 300 KiB

In [ ]:
#| export
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable

import h5py
import numpy as np

from stable_marl.data.dataset import Dataset, get_cache_dir, to_tensor
from stable_marl.data.format import Format, register_format, validate_write_mode

In [ ]:
#| export
class HDF5Dataset(Dataset):
    """
    Dataset reading one HDF5 file (opened lazily, so it works with DataLoader workers).

    Parameters
    ----------
    name : str
        Dataset name, read from ``<cache dir>/datasets/<name>.h5`` (see :func:`get_cache_dir`)
    frameskip, num_steps, transform
        See :class:`Dataset`
    keys_to_load : list of str
        Columns to load (default: all)
    keys_to_cache : list of str
        Columns read into memory once
    path : str or Path
        Path of the file (instead of `name`)
    """
    def __init__(
        self,
        name: str | None = None,
        frameskip: int = 1,
        num_steps: int = 1,
        transform: Callable[[dict], dict] | None = None,
        keys_to_load: list[str] | None = None,
        keys_to_cache: list[str] | None = None,
        cache_dir: str | Path | None = None,
        path: str | Path | None = None):
        if path is None:
            if name is None:
                raise TypeError("HDF5Dataset requires either `name` or `path`")
            path = get_cache_dir(cache_dir, sub_folder='datasets') / f'{name}.h5'
        self.h5_path = Path(path)
        self.h5_file: h5py.File | None = None
        self._cache: dict[str, np.ndarray] = {}
        with h5py.File(self.h5_path, 'r') as f:
            lengths, offsets = f['ep_len'][:], f['ep_offset'][:]
            self._keys = keys_to_load or [k for k in f.keys() if k not in ('ep_len', 'ep_offset')]
            self.metadata = {k: _decode_attr(v) for k, v in f.attrs.items()}
            for key in keys_to_cache or []:
                self._cache[key] = f[key][:]
        super().__init__(lengths, offsets, frameskip, num_steps, transform)

    @property
    def column_names(self) -> list[str]:
        return self._keys

    def _open(self):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r', swmr=True, rdcc_nbytes=256 * 1024 * 1024)

    def close(self):
        "Close the file (it is re-opened by the next read)."
        if self.h5_file is not None:
            self.h5_file.close()
            self.h5_file = None

    def __getstate__(self) -> dict:
        state = self.__dict__.copy()
        state['h5_file'] = None
        return state

    def _load_slice(self, ep_idx: int, start: int, end: int) -> dict:
        self._open()
        g_start, g_end = self.offsets[ep_idx] + start, self.offsets[ep_idx] + end
        steps = {}
        for col in self._keys:
            data = (self._cache if col in self._cache else self.h5_file)[col][g_start:g_end]
            if col != 'action':
                data = data[::self.frameskip]
            if data.dtype == np.object_ or data.dtype.kind in ('S', 'U'):
                value = data[0] if len(data) else b''
                steps[col] = value.decode() if isinstance(value, bytes) else value
            else:
                steps[col] = to_tensor(data)
        return self.transform(steps) if self.transform else steps

    def get_col_data(self, col: str) -> np.ndarray:
        if col in self._cache:
            return self._cache[col]
        self._open()
        return self.h5_file[col][:]

    def get_row_data(self, row_idx: int | list[int]) -> dict:
        self._open()
        return {col: self.h5_file[col][row_idx] for col in self._keys}


def _decode_attr(value: Any) -> Any:
    if isinstance(value, bytes):
        value = value.decode()
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value.item() if isinstance(value, np.generic) else value

In [ ]:
#| export
def _is_string_col(sample) -> bool:
    if isinstance(sample, (str, bytes, bytearray)):
        return True
    return np.asarray(sample).dtype.kind in ('U', 'S', 'O')


class HDF5Writer:
    """
    Appends episodes to one HDF5 file. The columns are taken from the first episode.

    Parameters
    ----------
    path : str or Path
        The ``.h5`` file
    mode : str
        ``'append'`` (default), ``'overwrite'`` or ``'error'`` if the file exists
    compression : str or None
        HDF5 compression of the columns (``'gzip'``, ``'lzf'`` (faster, larger) or None)

    ``metadata`` (a dict) is written as file attributes when the writer closes.
    """
    def __init__(self, path, *, mode: str = 'append', compression: str | None = 'gzip'):
        validate_write_mode(mode)
        self.path, self.mode, self.compression = Path(path), mode, compression
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.metadata: dict[str, Any] = {}
        self._f: h5py.File | None = None
        self._initialized = self._appending_existing = False
        self._ep_written = self._global_ptr = 0

    def __enter__(self) -> HDF5Writer:
        exists = self.path.exists()
        if exists and self.mode == 'error':
            raise FileExistsError(f"HDF5Writer: '{self.path}' already exists. Pass mode='overwrite' "
                                  "to replace it or mode='append' to extend it.")
        if self.mode == 'overwrite' or not exists:
            self._f = h5py.File(self.path, 'w', libver='latest')
        else:
            self._f = h5py.File(self.path, 'a', libver='latest')
            self._load_existing_state()
        return self

    def __exit__(self, *exc):
        if self._f is not None:
            for key, value in self.metadata.items():
                self._f.attrs[key] = value if isinstance(value, (int, float, str)) else json.dumps(value, default=str)
            self._f.close()
            self._f = None

    def write_episode(self, ep_data: dict):
        if self._f is None:
            raise RuntimeError("HDF5Writer used outside of a `with` block")
        if not self._initialized:
            self._init_schema(ep_data)
            self._initialized = True
        elif self._appending_existing and self._ep_written == 0:
            self._validate_episode_against_existing(ep_data)

        ep_len = len(next(iter(ep_data.values())))
        for col, values in ep_data.items():
            ds = self._f[col]
            ds.resize(self._global_ptr + ep_len, axis=0)
            if h5py.check_string_dtype(ds.dtype):
                values = np.array([v.decode() if isinstance(v, (bytes, bytearray)) else str(v) for v in values], dtype=object)
            ds[self._global_ptr:self._global_ptr + ep_len] = np.asarray(values)

        n = self._f['ep_len'].shape[0]
        for key, value in (('ep_len', ep_len), ('ep_offset', self._global_ptr)):
            self._f[key].resize(n + 1, axis=0)
            self._f[key][n] = value
        self._ep_written += 1
        self._global_ptr += ep_len

    def write_episodes(self, episodes):
        for episode in episodes:
            self.write_episode(episode)

    def _load_existing_state(self):
        if 'ep_len' not in self._f or 'ep_offset' not in self._f:
            raise ValueError(f"HDF5Writer: cannot append to '{self.path}', it has no ep_len / ep_offset")
        self._global_ptr = int(self._f['ep_len'][:].sum())
        self._initialized = self._appending_existing = True

    def _validate_episode_against_existing(self, ep_data: dict):
        existing = {k for k in self._f.keys() if k not in ('ep_len', 'ep_offset')}
        missing, extra = existing - set(ep_data), set(ep_data) - existing
        if missing or extra:
            raise ValueError(f"HDF5Writer: append failed, schema mismatch on '{self.path}'. "
                             f"Missing columns: {sorted(missing)}; unexpected columns: {sorted(extra)}.")
        for col, values in ep_data.items():
            if h5py.check_string_dtype(self._f[col].dtype):
                continue
            if np.asarray(values[0]).shape != self._f[col].shape[1:]:
                raise ValueError(f"HDF5Writer: append failed, column '{col}' has per-step shape "
                                 f"{self._f[col].shape[1:]} in the file and {np.asarray(values[0]).shape} here.")

    def _init_schema(self, sample_ep: dict):
        for col, values in sample_ep.items():
            if _is_string_col(values[0]):
                self._f.create_dataset(col, shape=(0,), maxshape=(None,), dtype=h5py.string_dtype(), chunks=(1,))
                continue
            sample = np.asarray(values[0])
            self._f.create_dataset(col, shape=(0, *sample.shape), maxshape=(None, *sample.shape),
                                   dtype=sample.dtype, chunks=(1, *sample.shape), compression=self.compression)
        self._f.create_dataset('ep_len', shape=(0,), maxshape=(None,), dtype=np.int32)
        self._f.create_dataset('ep_offset', shape=(0,), maxshape=(None,), dtype=np.int64)

In [ ]:
#| export
@register_format
class HDF5(Format):
    "A single ``.h5`` file (or a folder holding exactly one)."
    name = 'hdf5'

    @classmethod
    def detect(cls, path) -> bool:
        p = Path(path)
        if p.suffix in ('.h5', '.hdf5'):
            return True
        return p.is_dir() and (any(p.glob('*.h5')) or any(p.glob('*.hdf5')))

    @classmethod
    def open_reader(cls, path, **kwargs) -> HDF5Dataset:
        p = Path(path)
        if p.is_dir():
            files = sorted(p.glob('*.h5')) + sorted(p.glob('*.hdf5'))
            if len(files) != 1:
                raise ValueError(f"Expected one .h5 / .hdf5 file in {p}, found {len(files)}: pass the file")
            p = files[0]
        return HDF5Dataset(path=p, **kwargs)

    @classmethod
    def open_writer(cls, path, **kwargs) -> HDF5Writer:
        return HDF5Writer(path, **kwargs)

### Tests

In [ ]:
import tempfile, torch
from fastcore.test import test_fail

def episode(T, A=2):
    return {'pixels': np.full((T, A, 4, 4, 3), T, np.uint8), 'state': np.arange(T)[:, None].repeat(3, 1),
            'action': np.arange(T * A, dtype=np.float32).reshape(T, A), 'step_idx': np.arange(T)}

with tempfile.TemporaryDirectory() as tmp:
    path = f'{tmp}/d.h5'
    with HDF5Writer(path) as w:
        w.metadata['env_name'] = 'X'
        w.metadata['env_kwargs'] = {'agents': 2}
        w.write_episodes([episode(5), episode(3)])
    ds = HDF5Dataset(path=path, num_steps=2)
    assert list(ds.lengths) == [5, 3] and list(ds.offsets) == [0, 5] and len(ds) == 4 + 2
    assert ds.metadata == {'env_name': 'X', 'env_kwargs': {'agents': 2}}
    item = ds[5]                                   # episode 1, start 1
    assert item['pixels'].shape == (2, 2, 4, 4, 3) and torch.equal(item['step_idx'], torch.tensor([1, 2]))
    assert item['action'].shape == (2, 2) and item['action'][0, 0] == 2
    assert ds.load_episode(0)['state'].shape == (5, 3) and ds.get_dim('pixels') == 96
    assert len(HDF5Dataset(path=path, num_steps=2, frameskip=2)) == 2   # span 4: only episode 0
    ds.close()                                                        # before writing to the file again

    with HDF5Writer(path) as w: w.write_episode(episode(2))          # append
    assert list(HDF5Dataset(path=path).lengths) == [5, 3, 2]
    def bad_append():
        with HDF5Writer(path) as w: w.write_episode({'pixels': episode(2)['pixels']})
    test_fail(bad_append, contains='schema mismatch')
    def error_mode():
        with HDF5Writer(path, mode='error'): pass
    test_fail(error_mode, contains='already exists')
    assert isinstance(HDF5.open_reader(tmp), HDF5Dataset) and HDF5.detect(path)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()